In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error
import time
import pygwalker as pyg
from datetime import datetime
import os
import csv

In [21]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

C:\Users\Dhvanish\AppData\Local\Temp\ipykernel_8536\13393110.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../artifacts/raw.csv')


In [22]:
df.isnull().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

## Imputing missing values ##

In [23]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [24]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - 
                                                                              df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [25]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - 
                                                                           df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


In [26]:
df.drop(['Store','Date','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [27]:
df.sample(10)

,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth
184004,1,0,0,0,1,0,0,a,a,40.0,1,2015,2,16,0,12.0,8,11.50,0
494659,1,11758,916,1,1,0,1,d,a,4580.0,0,2014,4,14,0,79.0,16,0.00,0
368508,2,6515,576,1,0,0,1,d,c,1500.0,0,2014,8,12,0,106.0,33,0.00,0
735065,3,11068,1097,1,1,0,0,a,a,3350.0,0,2013,9,11,1,8.0,37,0.00,0
15515,6,9498,1086,1,0,0,0,a,a,1080.0,0,2015,7,18,0,50.0,29,0.00,0
510312,1,8596,670,1,1,0,0,d,a,2960.0,1,2014,3,31,0,0.0,14,0.00,1
390473,7,0,0,0,0,0,0,a,a,90.0,0,2014,7,20,0,1.0,29,0.00,0
464866,7,0,0,0,0,0,0,d,a,3780.0,1,2014,5,11,1,16.0,19,30.75,0
965594,6,7581,749,1,0,0,0,d,c,9070.0,0,2013,2,16,0,110.0,7,0.00,0
325146,7,0,0,0,0,0,0,d,c,29070.0,0,2014,9,28,0,113.0,39,0.00,0


In [28]:
walker = pyg.walk(df)

Box(children=(HTML(value='\n<div id="ifr-pyg-00065a7f5174ba55lgntBG7hjYixN4AQ" style="height: auto">\n    <hea…

In [29]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [30]:
df['IsSunday'] = (df['DayOfWeek'] == 7).astype(int)

In [31]:
df['IsStoreType_b'] = (df['StoreType'] == 'b').astype(int)

In [32]:
#df.drop(columns=['StoreType','DayOfWeek'],inplace=True,axis=1)

In [33]:
df.sample(20)

,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,...,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth,IsSunday,IsStoreType_b
367489,3,4658,526,1,0,0,1,c,c,4630.0,...,2014,8,13,0,41.0,33,35.50,1,0,0
829908,1,8710,767,1,1,0,0,a,c,3240.0,...,2013,6,17,1,5.0,25,24.75,1,0,0
627861,1,6919,823,1,1,0,0,a,a,310.0,...,2013,12,16,0,0.0,51,0.00,0,0,0
898807,3,7925,629,1,0,0,0,d,c,7290.0,...,2013,4,17,0,0.0,16,0.00,0,0,0
787532,4,3446,303,1,0,0,0,d,a,310.0,...,2013,7,25,1,6.0,30,44.25,0,0,0
800506,7,0,0,0,0,0,1,a,a,50.0,...,2013,7,14,1,6.0,28,0.00,0,1,0
798103,2,4255,472,1,1,0,1,a,c,2330.0,...,2013,7,16,1,6.0,29,0.00,0,0,0
356230,1,5849,617,1,0,0,0,a,c,15050.0,...,2014,8,25,0,71.0,35,0.00,0,0,0
508919,2,9489,854,1,1,0,0,c,c,31830.0,...,2014,4,1,0,49.0,14,0.00,0,0,0
554406,4,7933,993,1,1,0,0,a,c,150.0,...,2014,2,20,1,13.0,8,0.00,0,0,0


In [39]:
num_col=['Customers','CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths']
cat_col = ['StoreType','Assortment','Year']

In [40]:
X = df.drop(columns=['Sales'])
y = df['Sales']

In [41]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [42]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first'),cat_col)    
])

In [43]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [44]:
results_file = 'rmsep_score.csv'
file_exists = os.path.isfile(results_file)

models = {
    "XGBRegressor" : XGBRegressor(tree_method='hist',n_jobs=-1),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=50,max_depth=15,n_jobs=-1),
    'LinearRegression': LinearRegression(),
    'LGBMRegressor': LGBMRegressor(n_jobs=-1)
}

results = []

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)

    training_start = time.perf_counter()
    model.fit(X_train,y_train)
    training_stop = time.perf_counter()
    training_time_taken = training_stop - training_start

    prediction_start = time.perf_counter()
    prediction = model.predict(X_test)
    prediction_stop = time.perf_counter()
    prediction_time_taken = prediction_stop-prediction_start
    rmse = root_mean_squared_error(y_test,prediction)
    rmsep = rmse/y_test_mean

    print(f'{model_name}: {rmsep*100:.2f}%\n')
    print(f'Training time taken for {model_name}: {training_time_taken:.4f}\n')
    print(f'prediction time taken for {model_name}: {prediction_time_taken:.4f}\n')

    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': model_name,
        'rmse': rmse,
        'rmsep_percent': rmsep * 100,
        'prediction_time_taken_sec': prediction_time_taken,
        'training_time_taken_sec': training_time_taken
    })

with open(results_file, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp', 'model_name', 'rmse', 'rmsep_percent', 'prediction_time_taken_sec','training_time_taken_sec'])
    if not file_exists:
        writer.writeheader()
    writer.writerows(results)

XGBRegressor: 13.47%

Training time taken for XGBRegressor: 5.8994

prediction time taken for XGBRegressor: 0.1767

RandomForestRegressor: 13.41%

Training time taken for RandomForestRegressor: 37.4222

prediction time taken for RandomForestRegressor: 0.4756

LinearRegression: 23.75%

Training time taken for LinearRegression: 0.3640

prediction time taken for LinearRegression: 0.0199

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012918 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1034
[LightGBM] [Info] Number of data points in the train set: 712046, number of used features: 11
[LightGBM] [Info] Start training from score 5774.019304
LGBMRegressor: 15.94%

Training time taken for LGBMRegressor: 2.4761

prediction time taken for LGBMRegressor: 0.2838

